# Foodture–FABIO L7 mapping

Produces `L7_meat_split` and `Summary_L7`.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


# Put the notebook and the two source files in the same folder.
DATA_DIR = Path.cwd()
INPUT_FILE = DATA_DIR / "Foodture_app.xlsx"
POPULATION_FILE = DATA_DIR / "population from 2015 to 2020.csv"
OUTPUT_FILE = DATA_DIR / "Foodture_results.xlsx"

## 1. Reusable functions

In [2]:
def weighted_profile(source, population, category, donor_countries):
    """Population-weighted L7 profile for one FoodEx2 L1 category."""
    country = source.index.get_level_values("country")
    l1 = source.index.get_level_values("L1")
    values = (
        source.loc[country.isin(donor_countries) & (l1 == category), ["g/capita"]]
        .groupby(level=["country", "L7"])["g/capita"]
        .sum()
        .unstack()
    )
    weights = population.reindex(values.index)
    return values.mul(weights, axis=0).sum().div(weights.sum()).to_frame("g/capita")


def regional_proxy(source, population, category, donors, targets, region):
    """Copy a weighted regional L7 profile to countries with missing detail."""
    profile = weighted_profile(source, population, category, donors)
    proxy = pd.concat({country: profile for country in targets}, names=["country"])
    proxy["L1"] = category
    proxy["L2"] = category
    proxy["% consumers"] = f"same as {region} EU"
    proxy["Source"] = f"same as {region} EU"
    return proxy.reset_index().set_index(["country", "L7", "L1", "L2"])


def mapped_rows(source, mapping, *, l1_name=None):
    """Select mapped L7 rows and attach their FABIO labels."""
    l7 = source.index.get_level_values("L7")
    mask = l7.isin(mapping)
    if l1_name is not None:
        mask &= source.index.get_level_values("L1") == l1_name
    rows = source.loc[mask, ["g/capita"]].copy()
    rows["FABIO"] = rows.index.get_level_values("L7").map(mapping)
    return rows


def aggregate_mapped(source, mapping, factors=None, *, l1_name=None):
    """Map L7 items, apply optional conversion factors, and aggregate."""
    rows = mapped_rows(source, mapping, l1_name=l1_name)
    if factors is None:
        factor = 1.0
    else:
        factor = rows.index.get_level_values("L7").map(factors)
    rows["g/capita_rev"] = rows["g/capita"] * factor
    return (
        rows.reset_index()
        .set_index(["country", "FABIO"])[["g/capita_rev"]]
        .groupby(level=["country", "FABIO"])
        .sum()
    )


def aggregate_l1(source, category, fabio_item, *, exclude=()):
    """Aggregate one L1 category to a single FABIO item."""
    l1 = source.index.get_level_values("L1")
    l7 = source.index.get_level_values("L7")
    rows = source.loc[(l1 == category) & ~l7.isin(exclude), ["g/capita"]].copy()
    rows["FABIO"] = fabio_item
    return (
        rows.reset_index()
        .set_index(["country", "FABIO"])[["g/capita"]]
        .rename(columns={"g/capita": "g/capita_rev"})
        .groupby(level=["country", "FABIO"])
        .sum()
    )


def recipe_rows(source, items, recipe, *, group=False):
    """Disaggregate selected L7 foods using fixed ingredient shares."""
    l7 = source.index.get_level_values("L7")
    consumption = source.loc[l7.isin(items), "g/capita"]
    ingredients = pd.concat(
        {fabio: consumption * share for fabio, share in recipe.items()}, axis=1
    )
    result = ingredients.stack().droplevel(["L7", "L1", "L2"]).to_frame("g/capita_rev")
    result.index.names = ["country", "FABIO"]
    if group:
        result = result.groupby(level=[0, 1]).sum()
        result.index.names = ["country", "FABIO"]
    return result


def grouped_recipe_rows(source, groups):
    """Reproduce the original grouped composite-dish allocation."""
    outputs = []
    l7 = source.index.get_level_values("L7")
    for _, items, recipe in groups:
        rows = source.loc[l7.isin(items), ["g/capita"]]
        total = rows.groupby(level="country")["g/capita"].sum()
        count = rows.groupby(level="country").size()
        ingredients = pd.concat(
            {fabio: total * count * share for fabio, share in recipe.items()}, axis=1
        )
        result = ingredients.stack().to_frame("g/capita_rev")
        result.index.names = ["country", "FABIO"]
        outputs.append(result)
    return pd.concat(outputs)

## 2. Load data and fill missing regional detail

In [3]:
population = (
    pd.read_csv(POPULATION_FILE, index_col=["Area", "Year"])[["Value"]]
    .unstack()
    .droplevel(0, axis=1)
    .rename(index={
        "Netherlands (Kingdom of the)": "Netherlands",
        "Czech Republic": "Czechia",
    })
    .mul(1e3)
    .loc[:, 2017:2020]
    .mean(axis=1)
)

consumer_raw = pd.read_excel(
    INPUT_FILE,
    sheet_name="Food consumption per country",
    index_col=[0, 1, 2, 3],
    header=3,
)
consumer_raw.index.names = ["country", "L7", "L1", "L2"]

IMITATE = "Products for non-standard diets, food imitates and food supplements"
COMPOSITE = "Composite dishes"

proxy_rules = [
    (IMITATE, ["Croatia", "Italy", "Slovenia", "Spain"], ["Greece", "Malta", "Cyprus"], "South"),
    (IMITATE, ["Hungary", "Poland"], ["Bulgaria", "Czechia", "Romania", "Slovakia"], "East"),
    (IMITATE, ["Estonia", "Finland", "Ireland", "Sweden"], ["Denmark", "Latvia", "Lithuania"], "North"),
    (IMITATE, ["Austria", "Belgium", "France", "Luxembourg", "Netherlands"], ["Germany"], "West"),
    (COMPOSITE, ["Croatia", "Slovenia", "Portugal"], ["Greece", "Italy", "Spain", "Malta", "Cyprus"], "South"),
    (COMPOSITE, ["Czechia", "Hungary", "Poland", "Romania", "Slovakia"], ["Bulgaria"], "East"),
]

proxies = [
    regional_proxy(consumer_raw, population, category, donors, targets, region)
    for category, donors, targets, region in proxy_rules
]

replace_pairs = {
    (country, category)
    for category, _, targets, _ in proxy_rules
    for country in targets
}
country = consumer_raw.index.get_level_values("country")
l1 = consumer_raw.index.get_level_values("L1")
replace_mask = np.fromiter(
    ((c, category) in replace_pairs for c, category in zip(country, l1)),
    dtype=bool,
    count=len(consumer_raw),
)
consumer_summary = pd.concat([consumer_raw.loc[~replace_mask], *proxies])

## 3. FoodEx2 L7 to FABIO mappings

### Meat

In [4]:
meat_mapping = {'Chicken fresh meat': 'Poultry Meat',
 'Turkey fresh meat': 'Poultry Meat',
 'Cooked turkey meat': 'Poultry Meat',
 'Chicken, minced meat': 'Poultry Meat',
 'Chicken, other slaughtering products': 'Poultry Meat',
 'Chicken feet': 'Poultry Meat',
 'Cured seasoned poultry meat': 'Poultry Meat',
 'Corned turkey': 'Poultry Meat',
 'Pig fresh meat': 'Pigmeat',
 'Pig minced meat': 'Pigmeat',
 'Ham, pork': 'Pigmeat',
 'Cooked pork ham': 'Pigmeat',
 'Cooked cured (or seasoned) pork meat': 'Pigmeat',
 'Cured seasoned pork meat': 'Pigmeat',
 'Bacon': 'Pigmeat',
 'Cured pork fat': 'Pigmeat',
 'Pig trotters and feet': 'Pigmeat',
 'Head cheese': 'Pigmeat',
 'Salami-type sausage': 'Pigmeat',
 'Sausages': 'Pigmeat',
 'Chipolata-type sausage': 'Pigmeat',
 'Bologna-type sausage': 'split bovine pig',
 'Ripened kolbasz': 'Pigmeat',
 'Cured ripened raw sausages': 'Pigmeat',
 'Sliceable or firm cooked sausages': 'split by national share',
 'Sliceable or firm cooked sausages ': 'split by national share',
 'Frankfurter sausage': 'Pigmeat',
 'Cooked salami': 'Pigmeat',
 'Polish-type cooked sausage': 'Pigmeat',
 'Wiener sausage': 'Pigmeat',
 'Mettwurst-type sausage': 'Pigmeat',
 'Mortadella-type sausage': 'Pigmeat',
 'Liver-type sausage': 'Pigmeat',
 'Blood-type sausage': 'Pigmeat',
 'Frankfurt-type sausage': 'Pigmeat',
 'Cooked sausages (generic)': 'split by national share',
 'Fresh spices sausages in casing': 'Pigmeat',
 'German salami': 'Pigmeat',
 'Hungarian-type salami': 'Pigmeat',
 'Breakfast-type sausage': 'split by national share',
 'Italian-style sausage': 'Pigmeat',
 'Fresh bratwurst': 'Pigmeat',
 'Blood and tongue sausage': 'Pigmeat',
 'Chorizo and similar': 'Pigmeat',
 'Fresh bulk sausages': 'Pigmeat',
 'Cured unripened raw sausages': 'Pigmeat',
 'Bovine fresh meat': 'Bovine Meat',
 'Cow, ox or bull fresh meat': 'Bovine Meat',
 'Calf fresh meat': 'Bovine Meat',
 'Bovine, minced meat': 'Bovine Meat',
 'Corned beef, cooked': 'Bovine Meat',
 'Bovine trotters and feet': 'Bovine Meat',
 'Lamb fresh meat': 'Mutton & Goat Meat',
 'Goat fresh meat': 'Mutton & Goat Meat',
 'Bovine edible offal, non-muscle, other than liver and kidney': 'Offals, Edible',
 'Pate, pork liver': 'Offals, Edible',
 'Liver based spreadable-textured specialities': 'Offals, Edible',
 'Blood (as part-nature)': 'Offals, Edible',
 'Bovine and pig fresh meat': 'split bovine pig',
 'Bovine and pig, minced meat': 'split bovine pig',
 'Processed whole meat products': 'split by national share',
 'Cooked cured (or seasoned) meat': 'split by national share',
 'Raw cured (or seasoned) meat': 'split by national share',
 'Canned meat': 'split by national share',
 'Mammals meat': 'split bovine pig',
 'Mammals and birds meat': 'split by national share',
 'Minced meat (as part-nature)': 'split by national share'}

L7_meat_rev = aggregate_mapped(
    consumer_summary,
    meat_mapping,
    l1_name="Meat and meat products",
)
split_meat = L7_meat_rev.index.get_level_values("FABIO").str.contains("split")
L7_meat_non_split = L7_meat_rev.loc[~split_meat]
L7_meat_split = L7_meat_rev.loc[split_meat]

### Fruit and vegetables

In [5]:
fruit_mapping = {'Apples': 'Apples and products',
 'Apples and similar': 'Apples and products',
 'Fruit compote, apple': 'Apples and products',
 'Bananas and similar': 'Bananas',
 'Common banana': 'Bananas',
 'Common banana ': 'Bananas',
 'Plantains': 'Plantains',
 'Oranges': 'Oranges, Mandarines',
 'Oranges, sweet': 'Oranges, Mandarines',
 'Mandarins': 'Oranges, Mandarines',
 'Lemons': 'Lemons, Limes and products',
 'Marmalade, oranges': 'Oranges, Mandarines',
 'Marmalade': 'Oranges, Mandarines',
 'Grapefruits': 'Grapefruit and products',
 'Table grapes': 'Grapes and products (excl wine)',
 'Table grapes ': 'Grapes and products (excl wine)',
 'Wine grapes': 'Grapes and products (excl wine)',
 'Dried vine fruits (raisins etc.)': 'Grapes and products (excl wine)',
 'Pineapples': 'Pineapples and products',
 'Canned or jarred pineapple': 'Pineapples and products',
 'Table olives': 'Olives (including preserved)',
 'Table olives ready for consumption': 'Olives (including preserved)',
 'Olives, processed': 'Olives (including preserved)',
 'Apricots': 'Fruits, Other',
 'Canned or jarred apricot': 'Fruits, Other',
 'Dried apricots': 'Fruits, Other',
 'Peaches and similar-': 'Fruits, Other',
 'Common peaches': 'Fruits, Other',
 'Nectarines': 'Fruits, Other',
 'Plums': 'Fruits, Other',
 'Jam, plums': 'Fruits, Other',
 'Damsons': 'Fruits, Other',
 'Cherries and similar-': 'Fruits, Other',
 'Canned or jarred sour cherry': 'Fruits, Other',
 'Strawberries': 'Fruits, Other',
 'Jam, strawberries': 'Fruits, Other',
 'Raspberries (red and yellow)': 'Fruits, Other',
 'Blackcurrants': 'Fruits, Other',
 'Bilberries (European blueberries)': 'Fruits, Other',
 'Berries and small fruits': 'Fruits, Other',
 'Figs': 'Fruits, Other',
 'Dried figs': 'Fruits, Other',
 'Prickly pears': 'Fruits, Other',
 'Avocados': 'Fruits, Other',
 'Kaki': 'Fruits, Other',
 'Pears': 'Fruits, Other',
 'Fuzzy kiwifruit': 'Fruits, Other',
 'Kiwi fruits (green, red, yellow)': 'Fruits, Other',
 'Canned or jarred fruit': 'Fruits, Other',
 'Canned or jarred mixed fruit': 'Fruits, Other',
 'Jam of fruit / vegetables': 'Fruits, Other',
 'Jam of fruit/vegetables': 'Fruits, Other',
 'Jam, apricots': 'Fruits, Other',
 'Jam, mixed fruit': 'Fruits, Other',
 'Compote of fruit / vegetables': 'Fruits, Other',
 'Fruit compote, mixed fruit': 'Fruits, Other',
 'Fruit compote, sour cherry': 'Fruits, Other',
 'Mixed dried fruits': 'Fruits, Other'}

vegetable_mapping = {'Tomatoes': 'Tomatoes and products',
 'Cherry tomatoes': 'Tomatoes and products',
 'Globe tomato': 'Tomatoes and products',
 'Tomato paste': 'Tomatoes and products',
 'Tomato puree': 'Tomatoes and products',
 'Preserved concentrated tomatoes': 'Tomatoes and products',
 'Preserved tomato, whole or pieces': 'Tomatoes and products',
 'Processed tomato products': 'Tomatoes and products',
 'Canned/jarred tomato sauce': 'Tomatoes and products',
 'Onions': 'Onions',
 'Green onions': 'Onions',
 'Spring onions': 'Onions',
 'Pearl onion': 'Onions',
 'Onion bulbs for fresh consumption': 'Onions',
 'Shallots and similar-': 'Onions'}

L7_fruit_rev = aggregate_mapped(
    consumer_summary,
    fruit_mapping,
    l1_name="Fruit and fruit products",
)
L7_Tomato_onion_rev = aggregate_mapped(consumer_summary, vegetable_mapping)
L7_veg_others_rev = aggregate_l1(
    consumer_summary,
    "Vegetables and vegetable products",
    "Vegetables, Other",
    exclude=vegetable_mapping,
)

### Grains and bakery products

In [6]:
Croissant_split = {'Croissant',
 'Croissant from puff pastry',
 'Croissant, filled with chocolate',
 'Croissant, filled with jam',
 'Doughnuts-berliner',
 'Kringles',
 'Pastry based on laminated dough',
 'Puff-pastry croissant, filled with chocolate',
 'Shortcrust (pies -tarts)',
 'Various pastry',
 'Yeast leavened pastry'}

Biscuits_split = {'Biscuit with inclusions, filling or coating',
 'Biscuits',
 'Biscuits, chocolate',
 'Biscuits, sweet, plain',
 'Biscuits, sweet, wheat wholemeal',
 'Butter biscuits',
 'Lebkuchen',
 'Wafers'}

cake_split = {'Cakes',
 'Cakes with fruits',
 'Chocolate cake',
 'Cream cake',
 'Fruit cake',
 'Muffins',
 'Plain cakes',
 'Sponge cake'}

pancake_split = {'Apple strudel',
 'Cream-cheese strudel',
 'Dumpling, sweet',
 'Flat bread-based pastry',
 'Fruit pie-tarts',
 'Pancakes',
 'Scones and similar',
 'Waffles'}

bun_split = {'Brioche type products',
 'Buns',
 'Fine yeast sweet dough (brioche)',
 'Laminated doughs',
 'Phyllo dough',
 'Short pastry dough (pate brisee)',
 'Suet pastry (pâte a grasse de boeuf) dough',
 'Unleavened doughs'}

grain_mapping = {'Wheat flour': 'Wheat and products',
 'Wheat flour white': 'Wheat and products',
 'Wheat semolina': 'Wheat and products',
 'Cereal and cereal-like flours': 'Wheat and products',
 'Bulgur': 'Wheat and products',
 'Rice grain': 'Rice and products',
 'Rice grain, long-grain': 'Rice and products',
 'Rice grain, polished': 'Rice and products',
 'Rice flour': 'Rice and products',
 'Rice and similar-': 'Rice and products',
 'Glass noodle': 'Rice and products',
 'Maize grain': 'Maize and products',
 'Maize meal': 'Maize and products',
 'Maize flour': 'Maize and products',
 'Maize semolina': 'Maize and products',
 'Barley grain, pearled': 'Barley and products',
 'Barley groats': 'Barley and products',
 'Buckwheat': 'Cereals, Other',
 'Pasta and similar products': 'Wheat and products',
 'Dried pasta': 'Wheat and products',
 'Dried egg pasta': 'Wheat and products',
 'Pasta, plain (not stuffed), uncooked': 'Wheat and products',
 'Dried durum pasta': 'Wheat and products',
 'Fresh egg pasta': 'Wheat and products',
 'Gnocchi': 'Potatoes and products',
 'Rye bread and rolls, refined flour': 'Rye and products',
 'Rye bread and rolls, wholemeal': 'Rye and products',
 'Rye only bread and rolls': 'Rye and products',
 'Cereal bars mixed': 'Cereals, Other',
 'Cereal flakes and similar': 'Cereals, Other',
 'Extruded breakfast cereal products': 'Cereals, Other',
 'Mixed breakfast cereals': 'Cereals, Other',
 'Muesli and similar mixed breakfast cereals': 'Cereals, Other',
 'Muesli breakfast cereals': 'Cereals, Other',
 'Oat rolled grains': 'Oats',
 'Oat rolled grains, wholemeal': 'Oats',
 'Popped cereals': 'Cereals, Other',
 'Processed maize-based flakes': 'Maize and products',
 'Processed mixed cereal-based flakes': 'Cereals, Other',
 'Processed oat-based flakes': 'Oats',
 'Processed wheat-based flakes': 'Wheat and products',
 'Rice, long-grain': 'Rice and products',
 'Rolled oats, instant': 'Oats'}

Rye_bread = {'Crisp bread, rye wholemeal',
 'Crisp bread, rye, refined flour',
 'Mixed wheat and rye bread and rolls',
 'Multigrain (not only rye-wheat) bread and rolls',
 'Rye-wheat bread and rolls, refined flour',
 'Rye-wheat bread and rolls, wholemeal'}

bakery_recipes = [
    (Croissant_split, {
        "Wheat and products": 0.35,
        "Sugar (Raw Equivalent)": 0.10,
        "Butter, Ghee": 0.25,
        "Milk - Excluding Butter": 0.15,
        "Eggs": 0.10,
        "Oilcrops Oil, Other": 0.05,
    }),
    (Biscuits_split, {
        "Wheat and products": 0.35,
        "Sugar (Raw Equivalent)": 0.30,
        "Milk - Excluding Butter": 0.05,
        "Butter, Ghee": 0.15,
        "Oilcrops Oil, Other": 0.15,
    }),
    (cake_split, {
        "Wheat and products": 0.30,
        "Sugar (Raw Equivalent)": 0.30,
        "Milk - Excluding Butter": 0.05,
        "Eggs": 0.15,
        "Butter, Ghee": 0.10,
        "Oilcrops Oil, Other": 0.10,
    }),
    (pancake_split, {
        "Wheat and products": 0.30,
        "Sugar (Raw Equivalent)": 0.05,
        "Milk - Excluding Butter": 0.40,
        "Eggs": 0.10,
        "Butter, Ghee": 0.10,
        "Oilcrops Oil, Other": 0.05,
    }),
    (bun_split, {
        "Wheat and products": 0.60,
        "Sugar (Raw Equivalent)": 0.10,
        "Milk - Excluding Butter": 0.05,
        "Eggs": 0.05,
        "Butter, Ghee": 0.10,
        "Oilcrops Oil, Other": 0.10,
    }),
]

grain_rows = consumer_summary.loc[
    consumer_summary.index.get_level_values("L1") == "Grains and grain-based products"
]
grain_l7 = grain_rows.index.get_level_values("L7")
rye_related = set(grain_l7[grain_l7.str.contains("rye", case=False)])
wheat_bread = set(
    grain_rows.loc[
        grain_rows.index.get_level_values("L2") == "Bread and similar products"
    ].index.get_level_values("L7")
) - rye_related

grain_parts = [
    *(recipe_rows(consumer_summary, items, recipe, group=True) for items, recipe in bakery_recipes),
    aggregate_mapped(consumer_summary, grain_mapping),
    recipe_rows(consumer_summary, wheat_bread, {
        "Wheat and products": 0.80,
        "Sugar (Raw Equivalent)": 0.05,
        "Milk - Excluding Butter": 0.05,
        "Oilcrops Oil, Other": 0.10,
    }),
    recipe_rows(consumer_summary, Rye_bread, {
        "Rye and products": 0.80,
        "Sugar (Raw Equivalent)": 0.05,
        "Milk - Excluding Butter": 0.05,
        "Oilcrops Oil, Other": 0.10,
    }),
]
L7_grain_summary = pd.concat(grain_parts).groupby(level=["country", "FABIO"]).sum()

### Potatoes, fats, oils, sugar, and confectionery

In [7]:
fat_oil_mapping = {'Butter': 'Butter, Ghee',
 'Butter and margarine/oil blends': 'Butter, Ghee',
 'Butter and margarine/oils blends': 'Butter, Ghee',
 'Pork lard': 'Fats, Animals, Raw',
 'Olive oil, refined': 'Olive Oil',
 'Olive oil, virgin or extra-virgin': 'Olive Oil',
 'Olive oils': 'Olive Oil',
 'Palm kernel oil, edible': 'Palmkernel Oil',
 'Peanut oil, edible': 'Groundnut Oil',
 'Rape seed oil, edible': 'Rape and Mustard Oil',
 'Sunflower seed oil, edible': 'Sunflowerseed Oil',
 'Grape seed oil': 'Oilcrops Oil, Other',
 'Pumpkin seed oil': 'Oilcrops Oil, Other',
 'Seed oils': 'Oilcrops Oil, Other',
 'Vegetable fats and oils, edible': 'Oilcrops Oil, Other',
 'Animal and vegetable fats/oils': 'Oilcrops Oil, Other',
 'Blended margarine': 'Palm Oil',
 'Fat emulsions and blended fats': 'Palm Oil',
 'Functional vegetable margarines/fats': 'Palm Oil',
 'Margarine omega 3, full fat': 'Palm Oil',
 'Margarins and similar': 'Palm Oil',
 'Traditional margarine': 'Palm Oil',
 'Shortening and similar baking fats': 'Palmkernel Oil',
 'Blended fat and oils': 'Palm Oil',
 'Blended frying oil/fats': 'Palm Oil',
 'Fries cooking oil': 'Oilcrops Oil, Other',
 'Vegetable savoury extracts': 'Oilcrops Oil, Other'}

sugar_mapping = {'Honey': 'Honey',
 'Honey, polyfloral': 'Honey',
 'Honey, blended': 'Honey',
 'Honey, monofloral': 'Honey',
 'White sugar': 'Sugar (Raw Equivalent)',
 'Sucrose (common sugar)': 'Sugar (Raw Equivalent)',
 'Refined beet sugar': 'Sugar beet',
 'Refined cane sugar': 'Sugar cane',
 'Brown sugar': 'Sugar (Raw Equivalent)',
 'Table-top sweeteners formulations': 'Sweeteners, Other',
 'Table-top sweeteners in tablets': 'Sweeteners, Other',
 'Syrups': 'Sweeteners, Other',
 'Milk chocolate': 'Chocolate',
 'Bitter chocolate': 'Chocolate',
 'Bitter-sweet chocolate': 'Chocolate',
 'White chocolate': 'Chocolate',
 'Chocolate coated confectionery': 'Chocolate',
 'Chocolate spread': 'Chocolate',
 'Cream chocolate': 'Chocolate',
 'Filled chocolate': 'Chocolate',
 'Pralines': 'Chocolate',
 'Chocolate and similar': 'Chocolate',
 'Chocolate/cocoa-based products': 'Chocolate',
 'Chocolate-based cakes': 'Chocolate',
 'Hot chocolate': 'Chocolate',
 'Gum drops': 'Sugar (Raw Equivalent)',
 'Chewing gum': 'Sugar (Raw Equivalent)',
 'Liquorice candies': 'Sugar (Raw Equivalent)',
 'Hard candies': 'Sugar (Raw Equivalent)',
 'Marzipan': 'Sugar (Raw Equivalent)',
 'Caramel, soft': 'Sugar (Raw Equivalent)',
 'Confectionery including chocolate': 'Sugar (Raw Equivalent)',
 'Jelly candies': 'Sugar (Raw Equivalent)',
 'Foamed sugar products (marshmallows)': 'Sugar (Raw Equivalent)',
 'Candies (soft and hard)': 'Sugar (Raw Equivalent)',
 'Gelatine dessert': 'Sugar (Raw Equivalent)'}

L7_potato_rev = aggregate_l1(
    consumer_summary,
    "Starchy roots or tubers and products thereof, sugar plants",
    "Potatoes and products",
)
L7_fat_oil_rev = aggregate_mapped(consumer_summary, fat_oil_mapping)

sugar = aggregate_mapped(consumer_summary, sugar_mapping)
is_chocolate = sugar.index.get_level_values("FABIO") == "Chocolate"
L7_sugar_non_cho = sugar.loc[~is_chocolate]
chocolate = sugar.loc[is_chocolate, "g/capita_rev"]
L7_sugar_cho = (
    pd.concat({
        "Sugar (Raw Equivalent)": chocolate * 0.50,
        "Cocoa and products": chocolate * 0.30,
        "Milk - Excluding Butter": chocolate * 1.15,
        "Oilcrops Oil, Other": chocolate * 0.05,
    }, axis=1)
    .stack()
    .droplevel("FABIO")
    .to_frame("g/capita_rev")
)
L7_sugar_cho.index.names = ["country", "FABIO"]

### Milk and dairy products

In [8]:
dairy_mapping = {'Milk': 'Milk - Excluding Butter',
 'Cow milk': 'Milk - Excluding Butter',
 'Cow milk, whole': 'Milk - Excluding Butter',
 'Cow milk, semi skimmed (half fat)': 'Milk - Excluding Butter',
 'Cow milk, semi-skimmed': 'Milk - Excluding Butter',
 'Cow milk, semi-skimmed (half fat)': 'Milk - Excluding Butter',
 'Cow milk, skimmed (low fat)': 'Milk - Excluding Butter',
 'Cow milk, Fat free': 'Milk - Excluding Butter',
 'Cow milk, full fat': 'Milk - Excluding Butter',
 'Cow milk, natural high fat': 'Milk - Excluding Butter',
 'Flavoured milks': 'Milk - Excluding Butter',
 'Evaporated milk (liquid, unsweetened)': 'Milk - Excluding Butter',
 'Yoghurt, cow milk, plain': 'Yoghurt',
 'Yoghurt, cow milk, flavoured': 'Yoghurt',
 'Yoghurt, cow milk': 'Yoghurt',
 'Yoghurt ': 'Yoghurt',
 'Strained yoghurt': 'Yoghurt',
 'Yoghurt drinks, including sweetened and/or flavoured variants': 'Yoghurt',
 'Kefir': 'Yoghurt',
 'Traditional sour milk products': 'Yoghurt',
 'Filmjölk': 'Yoghurt',
 'Viili': 'Yoghurt',
 'Buttermilk': 'Yoghurt',
 'Cultured buttermilk': 'Yoghurt',
 'Cheese': 'Cheese',
 'Mozzarella': 'Cheese',
 'Cheese, gouda': 'Cheese',
 'Cheese, parmigiano reggiano': 'Cheese',
 'Cheese, emmental': 'Cheese',
 'Cheese, edam': 'Cheese',
 'Cheese, cheddar': 'Cheese',
 'Cheese, feta': 'Cheese',
 'Cheese, brie': 'Cheese',
 'Cheese, camembert': 'Cheese',
 'Cheese, gruyere': 'Cheese',
 'Cheese, comte': 'Cheese',
 'Cheese, pecorino romano': 'Cheese',
 'Cheese, grana padano': 'Cheese',
 'Cheese, raclette': 'Cheese',
 'Cheese, halloumi': 'Cheese',
 'Cheese, mizithra': 'Cheese',
 'Cheese, danbo': 'Cheese',
 'Cheese, cream havarti': 'Cheese',
 'Cheese, oltermanni': 'Cheese',
 'Cheese, turunmaa': 'Cheese',
 'Cheese, kefalotyri': 'Cheese',
 'Cheese, trappist': 'Cheese',
 'Cheese, telemea': 'Cheese',
 'Cheese, caciocavallo': 'Cheese',
 'Cheese curd': 'Cheese',
 'Firm/semi-hard cheese (gouda and edam type)': 'Cheese',
 'Soft – ripened cheese': 'Cheese',
 'Fresh uncured cheese': 'Cheese',
 'Soft brined cheese (feta type)': 'Cheese',
 'Firm brined cheese (ricotta salata type)': 'Cheese',
 'Firm - ripened cheeses': 'Cheese',
 'Firm-ripened cheeses': 'Cheese',
 'Hard cheese (cheddar, emmental type)': 'Cheese',
 'Soft-ripened cheese veined with blue mould (blue bavarian, blue de graven type )': 'Cheese',
 'Cheese cake': 'Cheese',
 'Cheese, buche de chevre': 'Cheese',
 'Cheese, gjetost': 'Cheese',
 'Cottage cheese': 'Cheese',
 'Cream cheese': 'Cheese',
 'Halloumi': 'Cheese',
 'Quark': 'Cheese',
 'Cream, plain': 'Cream',
 'Sour cream, plain': 'Cream',
 'Créme fraiche and other mild variants of sour cream': 'Cream',
 'Flavoured sour cream': 'Cream',
 'Cream with other ingredients': 'Cream',
 'Clotted cream': 'Cream',
 'Ice cream, milk-based': 'Dessert',
 'Dairy desserts spoonable': 'Dessert',
 'Custard': 'Dessert',
 'Starchy pudding': 'Dessert',
 'Dairy snacks': 'Dessert',
 'Processed cheese, spreadable': 'Cheese',
 'Processed cheese and spreads': 'Cheese',
 'Processed cheese wedges and similar': 'Cheese',
 'Processed cheese, sliceable': 'Cheese',
 'Butter': 'Butter, Ghee'}

dairy_rows = mapped_rows(consumer_summary, dairy_mapping)
dairy_rows["factor"] = dairy_rows["FABIO"].map({"Cream": 3, "Cheese": 10}).fillna(1)

non_dessert = dairy_rows.loc[~dairy_rows["FABIO"].isin(["Dessert", "Yoghurt"])].copy()
non_dessert["g/capita_rev"] = non_dessert["g/capita"] * non_dessert["factor"]
non_dessert["FABIO"] = non_dessert["FABIO"].replace({
    "Cheese": "Milk - Excluding Butter",
    "Cream": "Milk - Excluding Butter",
})
L7_dairy_non_dessert_rev = (
    non_dessert.reset_index()
    .set_index(["country", "FABIO"])[["g/capita_rev"]]
    .groupby(level=["country", "FABIO"])
    .sum()
)

dessert_items = {item for item, fabio in dairy_mapping.items() if fabio == "Dessert"}
yoghurt_items = {item for item, fabio in dairy_mapping.items() if fabio == "Yoghurt"}
L7_dairy_dessert = recipe_rows(consumer_summary, dessert_items, {
    "Sugar (Raw Equivalent)": 0.20,
    "Milk - Excluding Butter": 0.70,
    "Oilcrops Oil, Other": 0.10,
})
L7_dairy_Yoghurt = recipe_rows(consumer_summary, yoghurt_items, {
    "Sugar (Raw Equivalent)": 0.15,
    "Milk - Excluding Butter": 0.85,
})

### Juices, legumes, eggs, and food imitates

In [9]:
juice_mapping = {'Juice, apple': 'Apples and products',
 'Nectar, apple': 'Apples and products',
 'Juice, orange': 'Oranges, Mandarines',
 'Juice concentrate, orange': 'Oranges, Mandarines',
 'Nectar, orange': 'Oranges, Mandarines',
 'Juice, lemon': 'Lemons, Limes and products',
 'Juice, grape': 'Grapes and products (excl wine)',
 'Juice, pineapple': 'Pineapples and products',
 'Juice, tomato': 'Tomatoes and products',
 'Nectar, peach': 'Fruits, Other',
 'Juice, apricot-orange': 'Fruits, Other',
 'Juice, mixed fruit': 'Fruits, Other',
 'Mixed fruit juice': 'Fruits, Other',
 'Mixed fruit nectars': 'Fruits, Other',
 'Fruit juice concentrates': 'Fruits, Other',
 'Fruit juices (100% from named source)': 'Fruits, Other',
 'Fruit nectars (min. 25-50% fruit as defined in EU legislation)': 'Fruits, Other',
 'Multivitamin juices': 'Fruits, Other',
 'Fruit smoothies': 'Fruits, Other',
 'Vegetable juices': 'Vegetables, Other',
 'Fruit / vegetable juices and nectars': 'Fruits, Other',
 'Fruit / vegetables juices and nectars': 'Fruits, Other'}

juice_ratio = {'Juice, apple': 1.4,
 'Nectar, apple': 0.7,
 'Juice, orange': 2.0,
 'Juice concentrate, orange': 2.0,
 'Nectar, orange': 1,
 'Juice, lemon': 2.0,
 'Juice, grape': 1.4,
 'Juice, pineapple': 1.3,
 'Juice, tomato': 1.5,
 'Nectar, peach': 0.7,
 'Juice, apricot-orange': 1.7,
 'Juice, mixed fruit': 1.7,
 'Mixed fruit juice': 1.7,
 'Mixed fruit nectars': 0.7,
 'Fruit juice concentrates': 1.7,
 'Fruit juices (100% from named source)': 1.7,
 'Fruit nectars (min. 25-50% fruit as defined in EU legislation)': 0.7,
 'Multivitamin juices': 1.7,
 'Fruit smoothies': 1.0,
 'Vegetable juices': 1.5,
 'Fruit / vegetable juices and nectars': 1.0,
 'Fruit / vegetables juices and nectars': 1.0}

legume_nut_spice_mapping = {'Canned or jarred chickpea': 'Pulses, Other and products',
 'Chickpeas (dry)': 'Pulses, Other and products',
 'Canned or jarred common beans': 'Pulses, Other and products',
 'Borlotti or other common beans (dry)': 'Pulses, Other and products',
 'Borlotti or other common beans (without pods)': 'Pulses, Other and products',
 'Kidney bean (dry seeds)': 'Pulses, Other and products',
 'Navy beans (dry seeds)': 'Pulses, Other and products',
 'Field beans (dry seeds)': 'Pulses, Other and products',
 'Canned or jarred legumes': 'Pulses, Other and products',
 'Lentils (dry)': 'Pulses, Other and products',
 'Garden peas (dry)': 'Peas',
 'Canned or jarred peas': 'Peas',
 'Peas (without pods) and similar-': 'Peas',
 'Peanuts': 'Groundnuts',
 'Peanut butter': 'Groundnuts',
 'Almonds': 'Nuts and products',
 'Almonds sweet': 'Nuts and products',
 'Walnuts': 'Nuts and products',
 'Hazelnuts': 'Nuts and products',
 'Chestnuts': 'Nuts and products',
 'Cashew nuts': 'Nuts and products',
 'Tree nuts': 'Nuts and products',
 'Dried nuts/seeds and related flours and powders': 'Nuts and products',
 'Sesame paste (tahini) (sesamus indicum)': 'Sesame seed',
 'Seed (as part-nature)': 'Oilcrops, Other',
 'Black caraway seed': 'Spices, Other',
 'Caraway fruit': 'Spices, Other',
 'Coriander seed': 'Spices, Other',
 'Cumin seed': 'Spices, Other',
 'Peppercorn (black, green and white)': 'Pepper',
 'Black pepper': 'Pepper',
 'Peppers, dried': 'Pepper',
 'Cloves buds': 'Cloves',
 'Nutmeg seed': 'Spices, Other',
 'Cinnamon bark': 'Spices, Other',
 'Allspice fruit': 'Spices, Other',
 'Ginger roots': 'Spices, Other',
 'Root and rhizome spices': 'Spices, Other',
 'Dried herbs': 'Spices, Other',
 'Paprika powder': 'Spices, Other',
 'Marjoram, dry': 'Spices, Other',
 'Majoram, dry': 'Spices, Other',
 'Bay leaves, dry': 'Spices, Other',
 'Mints, dry': 'Spices, Other',
 'Savory, dry': 'Spices, Other',
 'Provence herbs': 'Spices, Other',
 'Spices': 'Spices, Other'}

egg_potato_mapping = {'Hen eggs': 'Eggs',
 'Hen egg yolk': 'Eggs',
 'Hen egg white': 'Eggs',
 'Hen egg mixed whole': 'Eggs',
 'Fried eggs': 'Eggs',
 'Boiled eggs': 'Eggs',
 'Hardened egg products': 'Eggs'}

alt_food_mapping = {'Yeast bread – pizza dough': 'Wheat and products',
 'Yeast bread - pizza dough': 'Wheat and products',
 'Soya drink': 'Soyabeans',
 'Oats drink': 'Oats',
 'Rice drink': 'Rice and products',
 'Soya yoghurt': 'Soyabeans',
 'Tofu': 'Soyabeans',
 'Imitation cream': 'Oilcrops Oil, Other',
 'Non dairy coffee creamer': 'Oilcrops Oil, Other',
 'Formulations containing special fatty acids (e.g. Omega-3, essential fatty acids)': 'Oilcrops '
                                                                                      'Oil, Other',
 'Mineral only supplements': None,
 'Vitamin only supplements': None,
 'Vitamin and mineral supplements': None,
 'Combination of vitamin and mineral only supplements': None,
 'Protein and amino acids supplements': None}

conversion_ratio_alt = {'Yeast bread – pizza dough': 0.83,
 'Yeast bread - pizza dough': 0.83,
 'Preparations for raising and flavouring home-made bakery': 0.0,
 'Soya drink': 0.11,
 'Tofu': 0.7,
 'Soya yoghurt': 0.11,
 'Oats drink': 0.11,
 'Rice drink': 0.11,
 'Imitation cream': 0.25,
 'Non dairy coffee creamer': 0.25,
 'Formulations containing special fatty acids (e.g. Omega-3, essential fatty acids)': 1.0,
 'Mineral only supplements': 0.0,
 'Vitamin only supplements': 0.0,
 'Vitamin and mineral supplements': 0.0,
 'Combination of vitamin and mineral only supplements': 0.0,
 'Protein and amino acids supplements': 0.0}

L7_juice_rev = aggregate_mapped(consumer_summary, juice_mapping, juice_ratio)
L7_legumes_rev = aggregate_mapped(consumer_summary, legume_nut_spice_mapping)
L7_egg_rev = aggregate_mapped(consumer_summary, egg_potato_mapping)
L7_imitiates_rev = aggregate_mapped(
    consumer_summary,
    alt_food_mapping,
    conversion_ratio_alt,
)

### Fish and seafood

In [10]:
L7_fish_rev = aggregate_l1(consumer_summary, "Fish and seafood", "Fish, Seafood")
L7_fish_rev_r = L7_fish_rev.sort_index().copy()

fish_proxy_rules = [
    (["Bulgaria", "Czechia", "Poland", "Slovakia"], ["Hungary", "Romania"]),
    (["Croatia", "Cyprus", "Greece", "Italy", "Malta", "Portugal", "Spain"], ["Slovenia"]),
]
for donors, targets in fish_proxy_rules:
    donor_values = L7_fish_rev.loc[donors, "g/capita_rev"].droplevel("FABIO")
    donor_weights = population.reindex(donors)
    regional_average = donor_values.mul(donor_weights).sum() / donor_weights.sum()
    for target in targets:
        L7_fish_rev_r.loc[(target, "Fish, Seafood"), "g/capita_rev"] = regional_average

### Seasonings and sauces

In [11]:
sauce_tomato = {'Tomato ketchup and related sauces',
 'Tomato-based cooking sauce',
 'Tomato-containing cooked sauces'}

sauce_cream = {'Hollandaise type sauce',
 'Mayonnaise sauce',
 'Mayonnaise, hollandaise and related sauces',
 'Mayonnaise, low fat',
 'Salad dressing',
 'Tartar sauce'}

sauce_white = {'Cream sauce', 'White sauces', 'Cheese sauce', 'Bechamel sauce'}

sauce_seasoning_mapping = {'Mustard and related sauces': 'Spices, Other',
 'Mustard, mild': 'Spices, Other',
 'Mustard, sweet': 'Spices, Other',
 'Mixed herbs and spices': 'Spices, Other',
 'Mixed herbs for couscous': 'Spices, Other',
 'Seasoning mixes': 'Spices, Other',
 'Curry powder': 'Spices, Other',
 'Tandoori': 'Spices, Other',
 'Pin pepper fruit a peppercorn': 'Pepper'}

conversion_ratio_sauces = {'Mustard and related sauces': 0.2,
 'Mustard, mild': 0.2,
 'Mustard, sweet': 0.2,
 'Mixed herbs and spices': 1.0,
 'Mixed herbs for couscous': 1.0,
 'Seasoning mixes': 0.8,
 'Curry powder': 1.0,
 'Tandoori': 0.8,
 'Pin pepper fruit a peppercorn': 1.0}

L7_sauce_tomato = recipe_rows(consumer_summary, sauce_tomato, {
    "Tomatoes and products": 0.85,
    "Sugar (Raw Equivalent)": 0.10,
    "Oilcrops Oil, Other": 0.05,
})
L7_sauce_cream = recipe_rows(consumer_summary, sauce_cream, {
    "Eggs": 0.10,
    "Sugar (Raw Equivalent)": 0.10,
    "Oilcrops Oil, Other": 0.70,
})
L7_sauce_white = recipe_rows(consumer_summary, sauce_white, {
    "Milk - Excluding Butter": 0.70,
    "Sugar (Raw Equivalent)": 0.02,
    "Oilcrops Oil, Other": 0.15,
})
L7_sauces_rev = aggregate_mapped(
    consumer_summary,
    sauce_seasoning_mapping,
    conversion_ratio_sauces,
)

### Beverages

In [12]:
softdrink_mapping = {'Soft drinks': 'Sugar (Raw Equivalent)',
 'Cola beverages, caffeinic': 'Sugar (Raw Equivalent)',
 'Cola-type drinks': 'Sugar (Raw Equivalent)',
 'Cola mix, flavoured cola': 'Sugar (Raw Equivalent)',
 'Soft drinks with minor amounts of fruits or flavours': 'Sugar (Raw Equivalent)',
 'Soft drink, flavoured, no fruit': 'Sugar (Raw Equivalent)',
 'Soft drink, lemon flavour': 'Sugar (Raw Equivalent)',
 'Soft drink, orange flavour': 'Sugar (Raw Equivalent)',
 'Soft drink, mixed flavours': 'Sugar (Raw Equivalent)',
 'Soft drink, cocktail without alcohol': 'Sugar (Raw Equivalent)',
 'Soft drink, with lemons or strawberry syrup': 'Sugar (Raw Equivalent)',
 'Soft drink, with lemons, carbonated': 'Sugar (Raw Equivalent)',
 'Soft drink, flavoured with herbs': 'Sugar (Raw Equivalent)',
 'Liquid drink bases (including concentrates and home-made preparations)': 'Sugar (Raw Equivalent)',
 'Fruit soft drink, orange': 'Sugar (Raw Equivalent)',
 'Fruit soft drink, mixed fruit': 'Sugar (Raw Equivalent)',
 'Soft drink, with fruit juice (fruit content below the minimum for nectars)': 'Sugar (Raw '
                                                                               'Equivalent)',
 'Energy drinks': 'Sugar (Raw Equivalent)',
 'Isotonic and sport drinks': 'Sugar (Raw Equivalent)',
 'Functional drinks': 'Sugar (Raw Equivalent)',
 'Diet soft drink with caffeine': 'None',
 'Diet soft drinks with flavours': 'None',
 'Diet soft drinks with fruit juice': 'None'}

softdrink_ratio = {'Soft drinks': 0.1,
 'Cola beverages, caffeinic': 0.1,
 'Cola-type drinks': 0.1,
 'Cola mix, flavoured cola': 0.1,
 'Soft drinks with minor amounts of fruits or flavours': 0.1,
 'Soft drink, flavoured, no fruit': 0.1,
 'Soft drink, lemon flavour': 0.1,
 'Soft drink, orange flavour': 0.1,
 'Soft drink, mixed flavours': 0.1,
 'Soft drink, cocktail without alcohol': 0.1,
 'Soft drink, with lemons or strawberry syrup': 0.1,
 'Soft drink, with lemons, carbonated': 0.1,
 'Soft drink, flavoured with herbs': 0.1,
 'Liquid drink bases (including concentrates and home-made preparations)': 0.1,
 'Fruit soft drink, orange': 0.1,
 'Fruit soft drink, mixed fruit': 0.1,
 'Soft drink, with fruit juice (fruit content below the minimum for nectars)': 0.1,
 'Energy drinks': 0.1,
 'Isotonic and sport drinks': 0.1,
 'Functional drinks': 0.1}

L7_softdrink_rev = aggregate_mapped(
    consumer_summary,
    softdrink_mapping,
    softdrink_ratio,
)
L7_softdrink_rev = L7_softdrink_rev.loc[
    L7_softdrink_rev.index.get_level_values("FABIO") != "None"
]

### Composite dishes

In [13]:
potato_dish_mapping = {'Prepared potato salad', 'Potato based dishes', 'Potato casserole'}

potato_chips_mapping = {'French fries from cut potato',
 'Fries (finger chips)',
 'Pan-fried potato',
 'Potato crisps from dough',
 'Potato crisps from potato slices',
 'Potato crisps or sticks',
 'Roesti'}

Veg_salard = {'Greek salad',
 'Mixed green salad',
 'Mixed vegetable salad',
 'Mixed vegetables salad',
 'Prepared salad',
 'Salads'}

Vegetable_cheese = {'Vegetables, gratinated', 'Ratatouille', 'Vegetable based dishes', 'Vegetables casserole'}

composite_direct_mapping = {'Porridge (ready to eat)': 'Cereals, Other',
 'Porridge milk based (ready to eat)': 'Cereals, Other',
 'Porridge water based (ready to eat)': 'Cereals, Other',
 'Oat porridge': 'Oats',
 'Rice porridge': 'Rice and products',
 'Mixed vegetables soup': 'Vegetables, Other',
 'Mixed vegetables soup, clear': 'Vegetables, Other',
 'Mixed vegetables soup, with puree or pieces': 'Vegetables, Other',
 'Mixed vegetables': 'Vegetables, Other',
 'Fruit soup': 'Fruits, Other',
 'Tomato soup': 'Tomatoes and products',
 'Potato puree from potatoes': 'Potatoes and products',
 'Potato puree': 'Potatoes and products',
 'Potato boiled': 'Potatoes and products',
 'Potatoes meal': 'Potatoes and products',
 'Fruit salad': 'Fruits, Other',
 'Fish and seafood based dishes': 'Fish, Seafood',
 'Egg based dishes': 'Eggs',
 'Dairy/egg soup': 'Eggs',
 'Meat stew': 'Meat, Other',
 'Legumes based dishes': 'Pulses, Other and products',
 'Legume (beans) soup': 'Pulses, Other and products',
 'Beans meal': 'Pulses, Other and products'}

conversion_ratio_composite = {'Porridge (ready to eat)': 0.2,
 'Porridge milk based (ready to eat)': 0.2,
 'Porridge water based (ready to eat)': 0.2,
 'Oat porridge': 0.2,
 'Rice porridge': 0.2,
 'Mixed vegetables soup': 0.4,
 'Mixed vegetables soup, clear': 0.4,
 'Mixed vegetables soup, with puree or pieces': 0.4,
 'Mixed vegetables': 1,
 'Fruit soup': 0.4,
 'Tomato soup': 0.4,
 'Fruit salad': 1,
 'Potato puree from potatoes': 0.9,
 'Potato puree': 0.9,
 'Potato boiled': 1,
 'Potatoes meal': 1,
 'Fish and seafood based dishes': 1,
 'Egg based dishes': 1,
 'Dairy/egg soup': 0.4,
 'Meat stew': 1,
 'Legumes based dishes': 1,
 'Legume (beans) soup': 0.4,
 'Beans meal': 1}

pasta_mix = {'Lasagna',
 'Pasta based dishes, cooked',
 'Pasta, cooked, with mixed sauce',
 'Pastas and rice (or other cereal) based dishes'}

pasta_plain = {'Pasta, cooked', 'Pasta, plain (not stuffed), cooked'}

rice_dish = {'Rice and vegetables meal',
 'Rice based dishes, cooked',
 'Rice meal',
 'Rice, meat, and vegetables meal',
 'Risotto'}

maize_soup = {'Corn semolina based thick soup'}

cereal_snacks = {'Mixed cereal-based snacks'}

Hamburger_dish = {'Hamburger with bread', 'Sausage roll', 'Bread- (or similar products) based dishes'}

L7_potato_salad_rev = recipe_rows(consumer_summary, potato_dish_mapping, {
    "Potatoes and products": 0.70,
    "Oilcrops Oil, Other": 0.10,
    "Eggs": 0.10,
    "Vegetables, Other": 0.10,
})
L7_chips_rev = recipe_rows(consumer_summary, potato_chips_mapping, {
    "Potatoes and products": 0.70,
    "Oilcrops Oil, Other": 0.30,
})
L7_Veg_salard_rev = recipe_rows(consumer_summary, Veg_salard, {
    "Vegetables, Other": 0.70,
    "Oilcrops Oil, Other": 0.20,
    "Milk - Excluding Butter": 1.00,
})
L7_Vegetable_cheese_rev = recipe_rows(consumer_summary, Vegetable_cheese, {
    "Vegetables, Other": 0.70,
    "Olive Oil": 0.20,
    "Milk - Excluding Butter": 1.00,
})
L7_composite_direct_rev = aggregate_mapped(
    consumer_summary,
    composite_direct_mapping,
    conversion_ratio_composite,
)

composite_l7 = consumer_summary.loc[
    consumer_summary.index.get_level_values("L1") == "Composite dishes"
].index.get_level_values("L7")
sandwich_dish = set(composite_l7[composite_l7.str.contains("Sandwich")])
pizza_dish = set(composite_l7[composite_l7.str.contains("Pizza")])

composite_groups = [
    ("Pasta_mix", pasta_mix, {
        "Wheat and products": 0.65,
        "Milk - Excluding Butter": 0.20,
        "Oilcrops Oil, Other": 0.10,
        "Vegetables, Other": 0.05,
    }),
    ("Pasta_plain", pasta_plain, {
        "Wheat and products": 0.85,
        "Oilcrops Oil, Other": 0.15,
    }),
    ("Rice_dish", rice_dish, {
        "Rice and products": 0.60,
        "Meat, Other": 0.20,
        "Vegetables, Other": 0.10,
        "Oilcrops Oil, Other": 0.10,
    }),
    ("Maize_soup", maize_soup, {
        "Maize and products": 0.20,
        "Vegetables, Other": 0.00,
        "Oilcrops Oil, Other": 0.00,
    }),
    ("cereal_snacks", cereal_snacks, {
        "Cereals, Other": 0.80,
        "Oilcrops Oil, Other": 0.20,
    }),
    ("Sandwich_dish", sandwich_dish, {
        "Wheat and products": 0.65,
        "Vegetables, Other": 0.10,
        "Meat, Other": 0.20,
        "Milk - Excluding Butter": 0.50,
    }),
    ("Hamburger_dish", Hamburger_dish, {
        "Wheat and products": 0.45,
        "Meat, Other": 0.40,
        "Milk - Excluding Butter": 0.50,
        "Vegetables, Other": 0.10,
    }),
    ("pizza_dish", pizza_dish, {
        "Wheat and products": 0.45,
        "Meat, Other": 0.15,
        "Milk - Excluding Butter": 2.50,
        "Vegetables, Other": 0.10,
        "Oilcrops Oil, Other": 0.05,
    }),
]
L7_Composite_rev = grouped_recipe_rows(consumer_summary, composite_groups)

## 4. Build and save the two final tables

In [14]:
summary_parts = {
    "Meat": L7_meat_non_split,
    "Fruit": L7_fruit_rev,
    "Tomato & Onion": L7_Tomato_onion_rev,
    "Vegetable Others": L7_veg_others_rev,
    "Grain": L7_grain_summary,
    "Potato": L7_potato_rev,
    "fat_oil": L7_fat_oil_rev,
    "sugar_non_cho": L7_sugar_non_cho,
    "Chocolate": L7_sugar_cho,
    "Yoghurt": L7_dairy_Yoghurt,
    "dairy": L7_dairy_non_dessert_rev,
    "dessert": L7_dairy_dessert,
    "juice": L7_juice_rev,
    "legumes": L7_legumes_rev,
    "Egg": L7_egg_rev,
    "Fish": L7_fish_rev_r,
    "Imitiates": L7_imitiates_rev,
    "Sauces": L7_sauces_rev,
    "sauce_cream": L7_sauce_cream,
    "sauce_white": L7_sauce_white,
    "sauce_tomato": L7_sauce_tomato,
    "softdrinks": L7_softdrink_rev,
    "Veg salard": L7_Veg_salard_rev,
    "Veg cheese": L7_Vegetable_cheese_rev,
    "chips": L7_chips_rev,
    "potato_salad": L7_potato_salad_rev,
    "Composite_direct": L7_composite_direct_rev,
    "Composite dish": L7_Composite_rev,
}
Summary_L7 = pd.concat(summary_parts, names=["source"])

In [ ]:
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    L7_meat_split.to_excel(writer, sheet_name="L7_meat_split", float_format="%.15e")
    Summary_L7.to_excel(writer, sheet_name="Summary_L7", float_format="%.15e")

print(f"Saved: {OUTPUT_FILE}")